In [ ]:
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from utils import build_and_save_abstention_tables

## Settings

In [ ]:
# Set display options for pandas
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [ ]:
DATASET_NAME_ORDER = [
    "BH_1",
    "DA",
    # "alkox",
    # "oer_plate_a",
    "p3ht",
    # "photo_pce10",
    # "photo_wf3",
    # "suzuki_edbo",
    "suzuki",
]
MODEL_W_PARAMS_ORDER = [
    "GPT-5 mini medium",
    "GPT-4.1 mini temp0.5",
    "GPT-4o mini temp0.5",
    "Claude Sonnet 4.5 temp0.5",
    "Claude Haiku 4.5 temp0.5",
    # "Claude 3.5 Haiku temp0.5",
]

In [ ]:
METHOD_DIRS = {
    "Simple Prompting": "01_simple_prompting",
    "Default-Abstention Prompting": "02_default_abstention_prompting",
    "Strict Prompting": "03_strict_prompting",
    "Extremely Strict Prompting": "04_extremely_strict_prompting",
    "More Information": "05_more_information",
    "Verbalized Confidence": "06_verbalized_confidence",
    "Self-Reflection": "07_self_reflection",
    "Consistency": "08_consistency",
    "LLM-Judge with GPT-5 mini": "09_02_llm_collaboration_gpt_5_mini",
    "LLM-Judge with Claude Sonnet 4.5": "09_03_llm_collaboration_claude_sonnet_4p5",
}

PROMPTING_METHODS = [
    "Simple Prompting",
    "Default-Abstention Prompting",
    "Strict Prompting",
    "Extremely Strict Prompting",
]

## Ensemble

In [ ]:
SPECIFIC_METHOD_DIRS = {
    "Simple Prompting": "01_simple_prompting",
    "Default-Abstention Prompting": "02_default_abstention_prompting",
    "Strict Prompting": "03_strict_prompting",
    "Extremely Strict Prompting": "04_extremely_strict_prompting",
    "More Information": "05_more_information",
    "Verbalized Confidence": "06_verbalized_confidence",
    "Self-Reflection": "07_self_reflection",
    "Consistency": "08_consistency",
    "LLM-Judge with GPT-5 mini": "09_02_llm_collaboration_gpt_5_mini",
    "LLM-Judge with Claude Sonnet 4.5": "09_03_llm_collaboration_claude_sonnet_4p5",
}

In [ ]:
with open("./01_simple_prompting/results_final/pair_result_dfs.pkl", "rb") as f:
    pair_result_simple_dfs = pickle.load(f)

In [ ]:
risk_coverage_curves_ensemble = []
risk_coverage_curve_results_ensemble = []

for dataset_name, model_w_params in pair_result_simple_dfs:
    print(f"dataset_name: {dataset_name}, model_w_params: {model_w_params}")

    # Combine results from all methods
    pair_result_df = pair_result_simple_dfs[(dataset_name, model_w_params)]
    pair_result_df = pair_result_df[["ID_A", "ID_B", "setup_true"]]

    for method_name, dir_name in list(SPECIFIC_METHOD_DIRS.items()):
        print(f"method_name: {method_name}")
        with open(f"./{dir_name}/results_final/pair_result_dfs.pkl", "rb") as f:
            pair_result_dfs = pickle.load(f)
        if method_name in PROMPTING_METHODS:
            pair_result_df_tmp = pair_result_dfs[(dataset_name, model_w_params)][
                [
                    "ID_A",
                    "ID_B",
                    "majority_answer",
                    "majority_answer_correct",
                    "abstention_rate",
                ]
            ].rename(
                columns={
                    "majority_answer": f"majority_answer_{method_name}",
                    "majority_answer_correct": f"majority_answer_correct_{method_name}",
                    "abstention_rate": f"uncertainty_{method_name}",
                }
            )
        else:
            pair_result_df_tmp = pair_result_dfs[(dataset_name, model_w_params)][
                [
                    "ID_A",
                    "ID_B",
                    "majority_answer",
                    "majority_answer_correct",
                    "uncertainty",
                ]
            ].rename(
                columns={
                    "majority_answer": f"majority_answer_{method_name}",
                    "majority_answer_correct": f"majority_answer_correct_{method_name}",
                    "uncertainty": f"uncertainty_{method_name}",
                }
            )
        pair_result_df = pair_result_df.merge(
            pair_result_df_tmp, on=["ID_A", "ID_B"], how="left"
        )

    # Ensemble
    for i in pair_result_df.index:
        setup_true = pair_result_df.loc[i, "setup_true"]
        answers = pair_result_df.loc[
            i,
            [
                f"majority_answer_{method_name}"
                for method_name in list(SPECIFIC_METHOD_DIRS.keys())
            ],
        ].to_numpy()
        if len(answers[answers == "C"]) == len(answers) or len(
            answers[answers == "Tie"]
        ) == len(answers):
            majority_answer = "C"
            majority_answer_correct = np.nan
        else:
            a_count = np.sum(answers == "A")
            b_count = np.sum(answers == "B")
            if a_count > b_count:
                majority_answer = "A"
            elif b_count > a_count:
                majority_answer = "B"
            else:
                majority_answer = "Tie"
            majority_answer_correct = (
                1
                if majority_answer == setup_true
                else (0.5 if majority_answer == "Tie" else 0)
            )
        pair_result_df.loc[i, "majority_answer"] = majority_answer
        pair_result_df.loc[i, "majority_answer_correct"] = majority_answer_correct
    pair_result_df["uncertainty"] = pair_result_df[
        [
            f"uncertainty_{method_name}"
            for method_name in list(SPECIFIC_METHOD_DIRS.keys())
        ]
    ].mean(axis=1)

    n_all = len(pair_result_df)

    valid_mask = (
        pair_result_df["majority_answer_correct"].notna()
        & pair_result_df["uncertainty"].notna()
        & np.isfinite(pair_result_df["uncertainty"])
    )

    tau_candidates = sorted(
        pair_result_df.loc[valid_mask, "uncertainty"].unique().tolist()
    )

    curve = [
        {
            "tau": -np.inf,
            "risk": 0.0,
            "coverage": 0.0,
        }
    ]
    for tau in tau_candidates:
        accepted_df = pair_result_df[
            valid_mask & (pair_result_df["uncertainty"] <= tau)
        ]
        n_accepted = len(accepted_df)

        risk = 1.0 - accepted_df["majority_answer_correct"].mean()
        coverage = n_accepted / n_all

        curve.append(
            {
                "tau": tau,
                "risk": risk,
                "coverage": coverage,
            }
        )

    # sort
    curve = sorted(curve, key=lambda x: x["coverage"])

    # remove duplicate coverage points
    deduped_curve = []
    seen_coverages = set()
    for point in curve:
        coverage_key = point["coverage"]
        if coverage_key not in seen_coverages:
            deduped_curve.append(point)
            seen_coverages.add(coverage_key)
    curve = deduped_curve

    # save risk-coverage curve points
    for point in curve:
        risk_coverage_curves_ensemble.append(
            {
                "dataset_name": dataset_name,
                "model_w_params": model_w_params,
                **point,
            }
        )

    positive_points = [point for point in curve if point["coverage"] > 0]

    # max coverage
    max_coverage = max([point["coverage"] for point in positive_points], default=0.0)

    # min coverage
    min_coverage = min([point["coverage"] for point in positive_points], default=np.nan)

    # area under risk-coverage curve (step integral)
    if len(positive_points) == 0:
        aurcc_step_integral = np.nan
    else:
        aurcc_step_integral = 0.0
        for i in range(1, len(curve)):
            x1 = curve[i - 1]["coverage"]
            x2, y2 = curve[i]["coverage"], curve[i]["risk"]
            aurcc_step_integral += (x2 - x1) * y2

    # penalized area under risk-coverage curve (step integral)
    penalty_risks = [(0.5, "0p5"), (1.0, "1p0")]
    penalized_aurcc_step_integral_dict = {}
    for penalty_risk, penalty_risk_tag in penalty_risks:
        if len(positive_points) == 0:
            penalized_aurcc_step_integral = penalty_risk
        else:
            unattained_coverage = max(0.0, 1.0 - max_coverage)
            penalized_aurcc_step_integral = (
                aurcc_step_integral + penalty_risk * unattained_coverage
            )

        penalized_aurcc_step_integral_dict[penalty_risk_tag] = (
            penalized_aurcc_step_integral
        )

    # C@R
    coverage_at_risk_0p2 = max(
        [point["coverage"] for point in positive_points if point["risk"] <= 0.2],
        default=0.0,
    )

    # R@C
    risk_at_coverage_0p8 = np.nan
    for point in positive_points:
        if np.isnan(risk_at_coverage_0p8) and point["coverage"] >= 0.8:
            risk_at_coverage_0p8 = point["risk"]

    risk_coverage_curve_results_ensemble.append(
        {
            "dataset_name": dataset_name,
            "model_w_params": model_w_params,
            "aurcc_step_integral": aurcc_step_integral,
            "penalized_aurcc_step_integral_0p5": penalized_aurcc_step_integral_dict[
                "0p5"
            ],
            "penalized_aurcc_step_integral_1p0": penalized_aurcc_step_integral_dict[
                "1p0"
            ],
            "max_coverage": max_coverage,
            "min_coverage": min_coverage,
            "coverage_at_risk_0p2": coverage_at_risk_0p2,
            "risk_at_coverage_0p8": risk_at_coverage_0p8,
        }
    )

risk_coverage_curves_ensemble_df = pd.DataFrame(risk_coverage_curves_ensemble)
risk_coverage_curve_results_ensemble_df = pd.DataFrame(
    risk_coverage_curve_results_ensemble
)

## Scores

In [ ]:
risk_coverage_curve_results_df = pd.DataFrame()
risk_coverage_curves_df = pd.DataFrame()

method_names_w_ensemble = []
for method_name, dir_name in list(METHOD_DIRS.items()):
    risk_coverage_curve_results_df_tmp = pd.read_csv(
        f"./{dir_name}/results_final/risk_coverage_curve_results.csv"
    )
    risk_coverage_curves_df_tmp = pd.read_csv(
        f"./{dir_name}/results_final/risk_coverage_curves.csv"
    )
    method_names_w_ensemble.append(method_name)

    risk_coverage_curve_results_df_tmp["method"] = method_name
    risk_coverage_curves_df_tmp["method"] = method_name

    risk_coverage_curve_results_df = pd.concat(
        [risk_coverage_curve_results_df, risk_coverage_curve_results_df_tmp],
        ignore_index=True,
    )
    risk_coverage_curves_df = pd.concat(
        [risk_coverage_curves_df, risk_coverage_curves_df_tmp],
        ignore_index=True,
    )


# Add ensemble results to the dataframes
risk_coverage_curve_results_ensemble_df["method"] = "Ensemble"
risk_coverage_curves_ensemble_df["method"] = "Ensemble"
risk_coverage_curve_results_df = pd.concat(
    [risk_coverage_curve_results_df, risk_coverage_curve_results_ensemble_df],
    ignore_index=True,
)
risk_coverage_curves_df = pd.concat(
    [risk_coverage_curves_df, risk_coverage_curves_ensemble_df],
    ignore_index=True,
)

method_names_w_ensemble.append("Ensemble")

In [ ]:
risk_coverage_curve_results_df = risk_coverage_curve_results_df.rename(
    columns={
        "aurcc_step_integral": "aurc",
        "penalized_aurcc_step_integral_0p5": "penalized_aurc_50",
        "penalized_aurcc_step_integral_1p0": "penalized_aurc_100",
        "coverage_at_risk_0p2": "coverage_at_risk_20",
        "risk_at_coverage_0p8": "risk_at_coverage_80",
        "max_coverage": "maximum_attainable_coverage",
        "min_coverage": "first_attainable_coverage",
    }
)

In [ ]:
SCORE_SPECS = {
    "aurc": "smaller_is_better",
    "penalized_aurc_50": "smaller_is_better",
    "penalized_aurc_100": "smaller_is_better",
    "coverage_at_risk_20": "larger_is_better",
    "risk_at_coverage_80": "smaller_is_better",
    "maximum_attainable_coverage": "larger_is_better",
    "first_attainable_coverage": "smaller_is_better",
}

output = build_and_save_abstention_tables(
    df=risk_coverage_curve_results_df,
    score_specs=SCORE_SPECS,
    method_order=method_names_w_ensemble,
    dataset_order=DATASET_NAME_ORDER,
    model_order=MODEL_W_PARAMS_ORDER,
    output_dir="./images/tables",
)


In [ ]:
display(output.main_mean_rank_table)
display(output.main_rank_sd_table)
display(output.main_mean_rank_with_sd_table)
display(output.main_na_count_table)

In [ ]:
display(output.by_metric["aurc"].sub_raw_table)
display(output.by_metric["aurc"].sub_rank_table)
display(output.by_metric["aurc"].original_na_mask)

In [ ]:
display(output.by_metric["penalized_aurc_50"].sub_raw_table)
display(output.by_metric["penalized_aurc_50"].sub_rank_table)
display(output.by_metric["penalized_aurc_50"].original_na_mask)

In [ ]:
display(output.by_metric["penalized_aurc_100"].sub_raw_table)
display(output.by_metric["penalized_aurc_100"].sub_rank_table)
display(output.by_metric["penalized_aurc_100"].original_na_mask)

In [ ]:
display(output.by_metric["coverage_at_risk_20"].sub_raw_table)
display(output.by_metric["coverage_at_risk_20"].sub_rank_table)
display(output.by_metric["coverage_at_risk_20"].original_na_mask)

In [ ]:
display(output.by_metric["risk_at_coverage_80"].sub_raw_table)
display(output.by_metric["risk_at_coverage_80"].sub_rank_table)
display(output.by_metric["risk_at_coverage_80"].original_na_mask)

In [ ]:
display(output.by_metric["maximum_attainable_coverage"].sub_raw_table)
display(output.by_metric["maximum_attainable_coverage"].sub_rank_table)
display(output.by_metric["maximum_attainable_coverage"].original_na_mask)

In [ ]:
display(output.by_metric["first_attainable_coverage"].sub_raw_table)
display(output.by_metric["first_attainable_coverage"].sub_rank_table)
display(output.by_metric["first_attainable_coverage"].original_na_mask)

## Risk-coverage curve

In [ ]:
DEFAULT_MARKERS = ["o", "s", "D", "^", "v", "P", "X", "*", "<", ">", "h"]
METHOD_MARKER_MAP = {
    method: marker for method, marker in zip(method_names_w_ensemble, DEFAULT_MARKERS)
}

METHOD_COLOR_MAP = {
    "Simple Prompting": "#0072B2",  # blue
    "Default-Abstention Prompting": "#E69F00",  # orange
    "Strict Prompting": "#009E73",  # green
    "Extremely Strict Prompting": "#D55E00",  # vermilion
    "More Information": "#56B4E9",  # sky blue
    "Verbalized Confidence": "#CC79A7",  # reddish purple
    "Self-Reflection": "#A6761D",  # brown/ochre
    "Consistency": "#00A6D6",  # cyan
    "LLM-Judge with GPT-5 mini": "#7B2CBF",  # purple
    "LLM-Judge with Claude Sonnet 4.5": "#6B8E23",  # olive green
    "Ensemble": "#909090",  # gray
}

In [ ]:
def plot_risk_coverage_curves(
    risk_coverage_curves_df,
    methods=None,
    output_tag="",
    method_order=method_names_w_ensemble,
    legend_ncol=3,
):
    if methods is None:
        methods = [
            m for m in method_order if m in risk_coverage_curves_df["method"].unique()
        ]
    else:
        methods = [m for m in method_order if m in methods]
    dataset_names = [
        d
        for d in DATASET_NAME_ORDER
        if d in risk_coverage_curves_df["dataset_name"].unique()
    ]
    model_names = [
        m
        for m in MODEL_W_PARAMS_ORDER
        if m in risk_coverage_curves_df["model_w_params"].unique()
    ]
    df_plot = risk_coverage_curves_df[
        risk_coverage_curves_df["method"].isin(methods)
    ].copy()

    n_row = len(model_names)
    n_col = len(dataset_names)
    fig, axes = plt.subplots(
        n_row,
        n_col,
        figsize=(n_col * 3, n_row * 3),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    legend_handle_dict = {}

    for i, model_w_params in enumerate(model_names):
        for j, dataset_name in enumerate(dataset_names):
            ax = axes[i, j]
            dfi = df_plot[
                (df_plot["dataset_name"] == dataset_name)
                & (df_plot["model_w_params"] == model_w_params)
            ]
            if not dfi.empty:
                sns.lineplot(
                    data=dfi,
                    x="coverage",
                    y="risk",
                    hue="method",
                    style="method",
                    hue_order=method_order,
                    style_order=method_order,
                    palette=METHOD_COLOR_MAP,
                    markers=METHOD_MARKER_MAP,
                    estimator=None,
                    lw=1.5,
                    markersize=8,
                    dashes=False,
                    alpha=0.6,
                    ax=ax,
                )
                for line in ax.lines:
                    line.set_linestyle("dashed")

                handles, labels = ax.get_legend_handles_labels()
                for handle in handles:
                    if hasattr(handle, "set_linestyle"):
                        handle.set_linestyle("dashed")
                for handle, label in zip(handles, labels):
                    if label in methods:
                        legend_handle_dict[label] = handle
                leg = ax.get_legend()
                if leg is not None:
                    leg.remove()
            ax.set_title(
                f"{dataset_name} - {model_w_params}",
                fontsize=10,
                y=0.9,
            )
            if j == 0:
                ax.set_ylabel("Risk", fontsize=12)
            else:
                ax.set_ylabel("")
            if i == n_row - 1:
                ax.set_xlabel("Coverage", fontsize=12)
            else:
                ax.set_xlabel("")
            ax.set_xlim(0, 1)
            ax.set_xticks(np.arange(0.0, 1.01, 0.1))
            ax.set_ylim(0, 1)
            ax.set_yticks(np.arange(0.0, 1.01, 0.1))
            ax.tick_params(axis="both", labelsize=10)
            ax.grid(True, linestyle="--", alpha=0.3)

    legend_methods = [m for m in methods if m in legend_handle_dict]
    legend_handles = [legend_handle_dict[m] for m in legend_methods]
    if legend_handles:
        fig.legend(
            legend_handles,
            legend_methods,
            loc="upper center",
            fontsize=10,
            bbox_to_anchor=(0.5, 1.06),
            frameon=True,
            ncol=min(legend_ncol, len(legend_methods)),
        )

    plt.tight_layout()
    plt.savefig(
        f"images/risk_coverage_curve_{output_tag}.png",
        format="png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.savefig(
        f"images/risk_coverage_curve_{output_tag}.pdf",
        format="pdf",
        bbox_inches="tight",
    )
    plt.savefig(
        f"images/risk_coverage_curve_{output_tag}.svg",
        format="svg",
        bbox_inches="tight",
    )
    plt.show()


In [ ]:
METHODS_ALL = method_names_w_ensemble
METHODS_EXPLICIT = [
    "Simple Prompting",
    "Default-Abstention Prompting",
    "Strict Prompting",
    "Extremely Strict Prompting",
]
METHODS_SELF_ASSESSMENT = [
    "More Information",
    "Verbalized Confidence",
    "Self-Reflection",
]
METHODS_CONSISTENCY_JUDGE_ENSEMBLE = [
    "Consistency",
    "LLM-Judge with GPT-5 mini",
    "LLM-Judge with Claude Sonnet 4.5",
    "Ensemble",
]

In [ ]:
plot_risk_coverage_curves(
    risk_coverage_curves_df,
    methods=METHODS_ALL,
    output_tag="all",
    legend_ncol=3,
)

In [ ]:
plot_risk_coverage_curves(
    risk_coverage_curves_df,
    methods=METHODS_EXPLICIT,
    output_tag="part_1",
    legend_ncol=1,
)

In [ ]:
plot_risk_coverage_curves(
    risk_coverage_curves_df,
    methods=METHODS_SELF_ASSESSMENT,
    output_tag="part_2",
    legend_ncol=1,
)


In [ ]:
plot_risk_coverage_curves(
    risk_coverage_curves_df,
    methods=METHODS_CONSISTENCY_JUDGE_ENSEMBLE,
    output_tag="part_3",
    legend_ncol=1,
)


In [ ]:
# # Plot risk-coverage curve (rows: dataset_name, cols: model_w_params)
# dataset_names = [d for d in DATASET_NAME_ORDER if d in risk_coverage_curves_df["dataset_name"].unique()]
# model_names = [m for m in MODEL_W_PARAMS_ORDER if m in risk_coverage_curves_df["model_w_params"].unique()]

# n_row = len(model_names)
# n_col = len(dataset_names)
# fig, axes = plt.subplots(n_row, n_col, figsize=(n_col * 3, n_row * 3), sharex=True, sharey=True)

# handles, labels = None, None
# for i, model_w_params in enumerate(model_names):
#     for j, dataset_name in enumerate(dataset_names):
#         ax = axes[i, j]
#         dfi = risk_coverage_curves_df[
#             (risk_coverage_curves_df["dataset_name"] == dataset_name)
#             & (risk_coverage_curves_df["model_w_params"] == model_w_params)
#         ]
#         sns.lineplot(
#             data=dfi,
#             x="coverage",
#             y="risk",
#             hue="method",
#             style="method",
#             hue_order=method_names_w_ensemble,
#             estimator=None,
#             lw=2,
#             markers=True,
#             markersize=8,
#             dashes=True,
#             alpha=0.5,
#             ax=ax,
#         )

#         if handles is None:
#             handles, labels = ax.get_legend_handles_labels()
#         leg = ax.get_legend()
#         if leg is not None:
#             leg.remove()

#         ax.set_title(f"{dataset_name} - {model_w_params}", fontsize=10, y=0.9)
#         if j == 0:
#             ax.set_ylabel("Risk", fontsize=12)
#         if i == n_row - 1:
#             ax.set_xlabel("Coverage", fontsize=12)
#         ax.set_xlim(0, 1)
#         ax.set_xticks(np.arange(0.0, 1.01, 0.1))
#         ax.set_ylim(0, 1)
#         ax.set_yticks(np.arange(0.0, 1.01, 0.1))
#         ax.tick_params(axis="both", labelsize=10)
#         ax.grid(True, linestyle="--", alpha=0.3)

# if handles is not None and labels is not None:
#     fig.legend(handles, labels, loc="upper center", fontsize=10, bbox_to_anchor=(0.5, 1.06), frameon=True, ncol=3)

# plt.tight_layout()
# plt.savefig("images/risk_coverage_curve.png", format="png", dpi=300, bbox_inches="tight")
# plt.savefig("images/risk_coverage_curve.pdf", format="pdf", bbox_inches="tight")
# plt.savefig("images/risk_coverage_curve.svg", format="svg", bbox_inches="tight")
# plt.show()

In [ ]:
# # Plot risk-coverage curve (rows: dataset_name, cols: model_w_params)
# dataset_names = ["DA", "suzuki"]
# model_names = ["GPT-5 mini medium", "Claude Sonnet 4.5 temp0.5"]

# n_row = len(model_names)
# n_col = len(dataset_names)
# fig, axes = plt.subplots(n_row, n_col, figsize=(n_col * 4, n_row * 4), sharex=True, sharey=True)

# handles, labels = None, None
# for i, model_w_params in enumerate(model_names):
#     for j, dataset_name in enumerate(dataset_names):
#         ax = axes[i, j]
#         dfi = risk_coverage_curves_df[
#             (risk_coverage_curves_df["dataset_name"] == dataset_name)
#             & (risk_coverage_curves_df["model_w_params"] == model_w_params)
#         ]
#         sns.lineplot(
#             data=dfi,
#             x="coverage",
#             y="risk",
#             hue="method",
#             style="method",
#             hue_order=method_names_w_ensemble,
#             estimator=None,
#             lw=2,
#             markers=True,
#             markersize=8,
#             dashes=True,
#             alpha=0.5,
#             ax=ax,
#         )

#         if handles is None:
#             handles, labels = ax.get_legend_handles_labels()
#         leg = ax.get_legend()
#         if leg is not None:
#             leg.remove()

#         ax.set_title(f"{dataset_name} - {model_w_params}", fontsize=10, y=0.9)
#         if j == 0:
#             ax.set_ylabel("Risk", fontsize=12)
#         if i == n_row - 1:
#             ax.set_xlabel("Coverage", fontsize=12)
#         ax.set_xlim(0, 1)
#         ax.set_xticks(np.arange(0.0, 1.01, 0.1))
#         ax.set_ylim(0, 1.0)
#         ax.set_yticks(np.arange(0.0, 1.01, 0.1))
#         ax.tick_params(axis="both", labelsize=10)
#         ax.grid(True, linestyle="--", alpha=0.3)

# if handles is not None and labels is not None:
#     fig.legend(handles, labels, loc="upper center", fontsize=10, bbox_to_anchor=(0.5, 1.12), frameon=True, ncol=3)

# plt.tight_layout()
# plt.savefig("images/risk_coverage_curve_part.png", format="png", dpi=300, bbox_inches="tight")
# plt.savefig("images/risk_coverage_curve_part.pdf", format="pdf", bbox_inches="tight")
# plt.savefig("images/risk_coverage_curve_part.svg", format="svg", bbox_inches="tight")
# plt.show()